# Phase 6: Infrastructure as Code & CI/CD Deployment
## Terraform Modules, State Management, and Automated Pipelines

**Time Estimate:** 10–12 hours | **Prerequisites:** Phases 1–5 completed

---

### What you'll learn
How to define, deploy, and manage the entire compliance system as code — Lambda functions, DynamoDB tables, API Gateway, Cognito, EventBridge — using Terraform modules. Plus CI/CD pipelines for automated testing and deployment.

### Study cross-references
| Concept | DDIA Chapter | DVA-C02 | System Design Interview |
|---------|-------------|---------|-------------------------|
| Reliability through automation | Ch. 1: Reliability | CloudFormation, CodeDeploy | Ch. 4: Rate Limiter |
| Consistency in distributed deploy | Ch. 9: Consistency | StackSets, multi-region | Ch. 6: Key-Value Store |
| Idempotent operations | Ch. 11: Stream Processing | terraform apply idempotency | — |
| Schema evolution in infra | Ch. 4: Encoding | CloudFormation updates | — |

### Documentation links
- [Terraform AWS Provider](https://registry.terraform.io/providers/hashicorp/aws/latest/docs)
- [Terraform Module Registry](https://registry.terraform.io/)
- [CloudFormation User Guide](https://docs.aws.amazon.com/AWSCloudFormation/latest/UserGuide/)
- [CloudFormation vs Terraform](https://docs.aws.amazon.com/prescriptive-guidance/latest/choose-iac-tool/)
- [GitHub Actions](https://docs.github.com/en/actions)
- [AWS CodePipeline](https://docs.aws.amazon.com/codepipeline/latest/userguide/)
- [Checkov IaC Security Scanner](https://www.checkov.io/1.Welcome/Quick%20Start.html)

---

## Part 1: Infrastructure as Code Theory

### 1.1 Why IaC for Compliance?

For compliance tools, IaC isn't just convenient — it's a **requirement**:
- **Audit trail:** Every infrastructure change is a Git commit with author, date, and review
- **Reproducibility:** Same config deployed N times = identical infrastructure (deterministic, like our PDF generator)
- **Disaster recovery:** Lost AWS account? Redeploy from Git in 10 minutes
- **Environment parity:** dev/staging/prod share the same module definitions, differing only in variables

### 1.2 DDIA Connection: Reliability Through Automation (Ch. 1)

Kleppmann (p. 6): *"Humans are known to be unreliable... The best systems combine the two — using automation for routine tasks and keeping humans in the loop for the unusual and creative tasks."*

IaC applies this directly:
- **Routine:** Creating Lambda functions, DynamoDB tables, IAM roles → automated by Terraform
- **Creative:** Designing the access pattern, choosing partition keys, setting TTLs → human decisions encoded in `.tf` files
- **Review:** `terraform plan` shows exactly what will change before applying — the infrastructure equivalent of a dry run

### 1.3 Immutable Infrastructure

```
Mutable (old):  EC2 running → SSH in → apt upgrade → pray it works
Immutable (us): Terraform defines desired state → apply → new resources created
                Need a change? Modify code → apply → old resources replaced
```

Our compliance system is fully serverless (Lambda, DynamoDB, API Gateway), which is inherently immutable — you can't SSH into a Lambda function. Each deployment creates a new version.

### 1.4 GitOps Workflow

```
1. Developer commits .tf changes to feature branch
2. GitHub Actions: terraform fmt -check + terraform validate
3. Pull request: terraform plan output posted as PR comment
4. Reviewer approves: checks the plan, not the code
5. Merge to main: terraform apply runs automatically
6. Production updated: monitoring confirms health
```

---

## Part 2: Exploring Our Terraform Modules

Instead of pasting HCL into code cells (which can't run), let's inspect the actual `terraform/` directory in our project. Every module we discuss exists as real `.tf` files you can read and modify.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

# Survey the terraform directory
terraform_dir = os.path.abspath('../../terraform')
print(f"Terraform directory: {terraform_dir}")
print(f"\nModule structure:")

for root, dirs, files in sorted(os.walk(terraform_dir)):
    # Skip hidden dirs
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    level = root.replace(terraform_dir, '').count(os.sep)
    indent = '  ' * level
    dirname = os.path.basename(root)
    tf_files = [f for f in files if f.endswith('.tf')]
    if tf_files:
        print(f"{indent}{dirname}/")
        for f in sorted(tf_files):
            filepath = os.path.join(root, f)
            size = os.path.getsize(filepath)
            print(f"{indent}  {f} ({size} bytes)")

In [ ]:
# Read the DynamoDB module — this is what stores our ControlAssessment objects
dynamodb_main = os.path.join(terraform_dir, 'modules', 'dynamodb', 'main.tf')
if os.path.exists(dynamodb_main):
    with open(dynamodb_main) as f:
        content = f.read()
    print("=== terraform/modules/dynamodb/main.tf ===")
    print(content[:2000])
    if len(content) > 2000:
        print(f"\n... ({len(content)} total chars)")
else:
    print("DynamoDB module not found — check terraform/modules/dynamodb/")

# Show how DynamoDB schema maps to our Python models
from src.models import ControlAssessment, CompliancePosture, DriftEvent
print("\n=== Connection: Terraform DynamoDB → Python models ===")
print(f"ControlAssessment fields that become DynamoDB attributes:")
for field_name, field_obj in ControlAssessment.__dataclass_fields__.items():
    print(f"  {field_name}: {field_obj.type}")

In [ ]:
# Read the Lambda module — this runs our ControlMappingEngine
lambda_main = os.path.join(terraform_dir, 'modules', 'lambda', 'main.tf')
if os.path.exists(lambda_main):
    with open(lambda_main) as f:
        content = f.read()
    print("=== terraform/modules/lambda/main.tf ===")
    print(content[:2000])
    if len(content) > 2000:
        print(f"\n... ({len(content)} total chars)")

# Show what the Lambda does in our pipeline
print("\n=== Lambda Functions in Our Architecture ===")
print("1. collector-lambda    → Runs src/collector/orchestrator.py → produces ScanResult")
print("2. mapper-lambda       → Runs src/mapper/engine.py → produces ControlAssessment[]")
print("3. pdf-lambda          → Runs src/evidence/pdf_generator.py → produces PDF → S3")
print("4. drift-lambda        → Runs src/drift/detector.py → produces DriftEvent[]")
print("5. api-lambda          → Serves assessments/posture/drift as JSON (Phase 5)")

In [ ]:
# Read the dev environment entry point
dev_main = os.path.join(terraform_dir, 'environments', 'dev', 'main.tf')
if os.path.exists(dev_main):
    with open(dev_main) as f:
        content = f.read()
    print("=== terraform/environments/dev/main.tf ===")
    print(content[:3000])
    if len(content) > 3000:
        print(f"\n... ({len(content)} total chars)")

# Show the environment variable pattern
print("\n=== Environment Variables: How Terraform → Lambda → Python ===")
print("""
In main.tf:
  environment_variables = {
    DYNAMODB_TABLE = module.controls_table.table_name    # → 'compliance-controls-dev'
    EVIDENCE_BUCKET = module.evidence_bucket.bucket_name  # → 'evidence-dev-123456'
    LOG_LEVEL = 'DEBUG'
  }

In Python (src/collector/orchestrator.py):
  import os
  TABLE_NAME = os.environ['DYNAMODB_TABLE']  # reads from Lambda environment
  BUCKET = os.environ['EVIDENCE_BUCKET']
""")

## Part 3: CloudFormation Comparison (DVA-C02)

The DVA-C02 exam tests **CloudFormation**, not Terraform. The concepts are identical but the syntax differs. Here's our DynamoDB table in both:

**Terraform:**
```hcl
resource "aws_dynamodb_table" "assessments" {
  name         = "compliance-assessments-${var.environment}"
  billing_mode = "PAY_PER_REQUEST"
  hash_key     = "PK"
  range_key    = "SK"

  attribute { name = "PK"; type = "S" }
  attribute { name = "SK"; type = "S" }
  attribute { name = "GSI1PK"; type = "S" }
  attribute { name = "GSI1SK"; type = "S" }

  global_secondary_index {
    name            = "GSI1"
    hash_key        = "GSI1PK"
    range_key       = "GSI1SK"
    projection_type = "ALL"
  }

  point_in_time_recovery { enabled = true }
}
```

**CloudFormation (equivalent):**
```yaml
AWSTemplateFormatVersion: '2010-09-09'
Resources:
  AssessmentsTable:
    Type: AWS::DynamoDB::Table
    Properties:
      TableName: !Sub "compliance-assessments-${Environment}"
      BillingMode: PAY_PER_REQUEST
      KeySchema:
        - AttributeName: PK
          KeyType: HASH
        - AttributeName: SK
          KeyType: RANGE
      AttributeDefinitions:
        - AttributeName: PK
          AttributeType: S
        - AttributeName: SK
          AttributeType: S
        - AttributeName: GSI1PK
          AttributeType: S
        - AttributeName: GSI1SK
          AttributeType: S
      GlobalSecondaryIndexes:
        - IndexName: GSI1
          KeySchema:
            - AttributeName: GSI1PK
              KeyType: HASH
            - AttributeName: GSI1SK
              KeyType: RANGE
          Projection:
            ProjectionType: ALL
      PointInTimeRecoverySpecification:
        PointInTimeRecoveryEnabled: true
```

**Key CloudFormation concepts for DVA-C02:**
- `!Ref` — Reference another resource's ID
- `!Sub` — String substitution with `${Variable}`
- `!GetAtt` — Get resource attribute (e.g., `!GetAtt Table.Arn`)
- **Change Sets** — Preview changes before applying (like `terraform plan`)
- **StackSets** — Deploy across multiple accounts/regions
- **Nested Stacks** — Like Terraform modules (reusable templates)
- **Drift Detection** — Check if resources have been modified outside CloudFormation

---

## Part 4: CI/CD Pipeline

### GitHub Actions Workflow

```yaml
# .github/workflows/terraform.yml
name: Terraform CI/CD

on:
  push:
    branches: [main]
    paths: ['terraform/**']
  pull_request:
    branches: [main]
    paths: ['terraform/**']

jobs:
  validate:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: hashicorp/setup-terraform@v2
      - run: terraform fmt -check -recursive terraform/
      - run: cd terraform/environments/dev && terraform init -backend=false && terraform validate

  plan:
    needs: validate
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: aws-actions/configure-aws-credentials@v4
        with:
          role-to-assume: ${{ secrets.AWS_ROLE_ARN }}
          aws-region: us-east-1
      - run: cd terraform/environments/dev && terraform init && terraform plan -out=tfplan

  apply:
    needs: plan
    if: github.ref == 'refs/heads/main'
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: aws-actions/configure-aws-credentials@v4
        with:
          role-to-assume: ${{ secrets.AWS_ROLE_ARN }}
          aws-region: us-east-1
      - run: cd terraform/environments/dev && terraform init && terraform apply -auto-approve
```

### DVA-C02: AWS-Native CI/CD

The exam tests CodePipeline + CodeBuild + CodeDeploy:

```
CodeCommit/GitHub → CodePipeline → CodeBuild (test + build) → CodeDeploy (deploy)
```

**CodeBuild** runs `terraform plan` and `pytest`:
```yaml
# buildspec.yml
version: 0.2
phases:
  install:
    commands:
      - pip install -r requirements.txt
  build:
    commands:
      - python -m pytest tests/ -v
      - terraform plan -out=tfplan
```

**CodeDeploy for Lambda** supports traffic shifting:
- `LambdaCanary10Percent5Minutes` — 10% traffic for 5 min, then 100%
- `LambdaLinear10PercentEvery1Minute` — 10% more every minute
- `LambdaAllAtOnce` — instant cutover (risky for production)

---

## Part 5: Terraform State Management

### DDIA Connection: Single Source of Truth (Ch. 9)

Terraform state is the **single source of truth** for what's deployed. Kleppmann (p. 321): *"If a system can only have one leader, that leader determines the authoritative state."*

Terraform state is a leader-based system:
- The state file in S3 is the leader
- DynamoDB provides the distributed lock (preventing concurrent `terraform apply`)
- If state and reality diverge, `terraform plan` shows the diff (like a replication lag)

```hcl
# Backend configuration
terraform {
  backend "s3" {
    bucket         = "compliance-terraform-state"
    key            = "prod/terraform.tfstate"
    region         = "us-east-1"
    encrypt        = true
    dynamodb_table = "terraform-locks"   # Distributed lock
  }
}
```

**What happens if state is lost?**
1. Terraform thinks nothing is deployed
2. `terraform apply` tries to create everything fresh → conflicts with existing resources
3. Recovery: `terraform import` each resource, or recreate from scratch

**What happens if two people run `terraform apply` simultaneously?**
1. First person acquires DynamoDB lock → proceeds
2. Second person gets "Lock held by..." error → must wait
3. This is the same distributed locking Kleppmann describes in Ch. 8

---

## Part 6: Monitoring & Observability

### DVA-C02 Connection: CloudWatch, X-Ray, Alarms

**CloudWatch Metrics** (what to monitor for our compliance system):
- Lambda: Invocations, Errors, Duration, ConcurrentExecutions, Throttles
- DynamoDB: ConsumedReadCapacityUnits, ConsumedWriteCapacityUnits, ThrottledRequests
- API Gateway: 4XXError, 5XXError, Latency, Count

**CloudWatch Alarms** (from our `terraform/modules/monitoring/`):
```hcl
resource "aws_cloudwatch_metric_alarm" "lambda_errors" {
  alarm_name          = "compliance-api-errors"
  comparison_operator = "GreaterThanThreshold"
  evaluation_periods  = 1
  metric_name         = "Errors"
  namespace           = "AWS/Lambda"
  period              = 300    # 5 minutes
  statistic           = "Sum"
  threshold           = 5     # Alert if >5 errors in 5 min
  alarm_actions       = [aws_sns_topic.alarms.arn]
}
```

**X-Ray** traces requests end-to-end:
```
API Gateway → Lambda (mapper) → DynamoDB query → Lambda (PDF) → S3 upload
                                                       ↓
                                X-Ray shows: DynamoDB took 12ms, S3 took 340ms
```

---

## Exercises

### Exercise 6.1: Read and Annotate a Terraform Module

Open `terraform/modules/dynamodb/main.tf` and annotate each resource block:
1. What `src/models` class does each DynamoDB attribute correspond to?
2. What GSI enables the "get all failed controls for a scan" query from Phase 5?
3. What happens if you change `billing_mode` from `PAY_PER_REQUEST` to `PROVISIONED`?
4. Why is `point_in_time_recovery` enabled? (Hint: FedRAMP requirement)

In [ ]:
# Exercise 6.1 starter: Read the DynamoDB module and map to src/ models
from src.models import ControlAssessment, CompliancePosture

print("ControlAssessment fields → DynamoDB attributes:")
print(f"  control_id → PK (partition key, e.g. 'CTRL#AC-2')")
print(f"  scan_id    → SK (sort key, e.g. 'SCAN#2024-01-15T10-00-00Z_abc123')")
print(f"  status     → attribute (ControlStatus.value, e.g. 'FAIL')")
print(f"  control_family → GSI1PK (e.g. 'FAM#AC')")

# YOUR TURN: Open the actual .tf file and map all attributes
dynamodb_vars = os.path.join(terraform_dir, 'modules', 'dynamodb', 'variables.tf')
if os.path.exists(dynamodb_vars):
    with open(dynamodb_vars) as f:
        print(f"\n=== DynamoDB module variables ===")
        print(f.read()[:1500])

### Exercise 6.2: Write a CloudFormation Template (DVA-C02 Practice)

Convert the Lambda module (`terraform/modules/lambda/main.tf`) to a CloudFormation YAML template. Include:
1. `AWS::Lambda::Function` with runtime, handler, memory, timeout
2. `AWS::IAM::Role` with the Lambda execution policy
3. `AWS::CloudWatch::Alarm` for error monitoring
4. Use `!Ref` and `!GetAtt` for cross-resource references

**This is directly exam-relevant.** The DVA-C02 gives you a scenario and asks which CloudFormation snippet is correct.

### Exercise 6.3: CI/CD Pipeline Design

Design a GitHub Actions workflow that:
1. On PR: runs `pytest tests/` AND `terraform plan`
2. Posts the test results and plan output as PR comments
3. On merge to main: runs `terraform apply` for dev, then staging, then prod (sequential)
4. On failure: sends SNS notification and rolls back

Draw the pipeline as a diagram. Then compare with AWS CodePipeline: which steps would use CodeBuild vs CodeDeploy?

### Exercise 6.4: Disaster Recovery Scenario

Your S3 terraform state bucket is accidentally deleted. Walk through:
1. What's the immediate impact? Can you still `terraform plan`?
2. How would you recover? (Hint: `terraform import`)
3. How many resources would you need to import for our system?
4. What prevention measure should you add? (Hint: S3 versioning + MFA delete)

**DDIA connection (Ch. 9):** This is a split-brain scenario. Reality (AWS resources) and the "leader" (state file) disagree. Kleppmann discusses how to resolve such conflicts.

### Exercise 6.5: Compare Environments

Read `terraform/environments/dev/main.tf` and design a `prod` version that differs in:
1. Lambda memory: 512MB (vs 256MB in dev)
2. Reserved concurrency: 100 (vs -1 unlimited in dev)
3. DynamoDB: provisioned billing with auto-scaling (vs PAY_PER_REQUEST in dev)
4. Monitoring: 5-error threshold (vs 50 in dev)
5. KMS: customer-managed keys (vs AWS-managed in dev)

How do you keep the module code identical while varying these settings? (Answer: Terraform variables and `.tfvars` files)

---

## Part 7: DDIA Deep Dive — Idempotency and Determinism (Ch. 11)

### Terraform Apply as an Idempotent Operation

Kleppmann (p. 478): *"An idempotent operation is one that you can perform multiple times, and it has the same effect as performing it once."*

`terraform apply` is idempotent by design:
- Run it once: creates 15 resources
- Run it again with no changes: "No changes. Your infrastructure matches the configuration."
- Run it after someone manually changes a security group: reverts the change to match the config

This is the same property we built into our pipeline:
- Running `ControlMappingEngine.assess_all_controls(scan)` twice on the same scan produces identical assessments
- Running `PDFReportGenerator.generate(assessments, posture, path)` twice produces the same PDF content
- Running `terraform apply` twice produces the same infrastructure

**Why idempotency matters for compliance:** If an auditor asks "can you prove this infrastructure matches your approved configuration?", you run `terraform plan` and show "No changes" — proof that reality matches the approved code.

### Schema Evolution in Infrastructure (DDIA Ch. 4)

Adding a new DynamoDB GSI is a schema migration. Terraform handles this like schema evolution:
- **Forward compatible:** New index can be added without breaking existing queries
- **Backward compatible:** Old code that doesn't use the new index continues to work
- **Rolling update:** Terraform adds the GSI while the table stays online (eventually consistent)

This mirrors Kleppmann's discussion of Avro schema evolution — new fields with defaults don't break old readers.

---

## Summary

### What you learned
- Explored the actual `terraform/` directory in our project
- Mapped Terraform resources to `src/models` Python classes
- Compared Terraform and CloudFormation (DVA-C02 exam prep)
- Designed CI/CD pipelines with GitHub Actions and AWS CodePipeline
- Understood state management as a distributed systems problem

### Pipeline — fully deployed
```
terraform/
├── modules/lambda/        → Deploys collector, mapper, PDF, drift, API Lambdas
├── modules/dynamodb/      → Stores ControlAssessment, CompliancePosture, DriftEvent
├── modules/api_gateway/   → Exposes REST API (Phase 5)
├── modules/cognito/       → Authentication for dashboard
├── modules/eventbridge/   → Scheduled scans (cron: rate(24 hours))
└── modules/monitoring/    → CloudWatch alarms, dashboards, X-Ray

.github/workflows/terraform.yml → Automated: validate → plan → apply
```

### Key classes → Terraform mapping
| `src/` Class | Terraform Resource | DynamoDB Key Pattern |
|-------------|-------------------|---------------------|
| `ControlAssessment` | `modules/dynamodb` | PK=`CTRL#{id}`, SK=`SCAN#{scan_id}` |
| `CompliancePosture` | `modules/dynamodb` | PK=`POSTURE`, SK=`SCAN#{scan_id}` |
| `DriftEvent` | `modules/dynamodb` | PK=`DRIFT`, SK=`SCAN#{scan_id}#{drift_id}` |
| `PDFReportGenerator` | `modules/lambda` + layer | Triggered by scan completion |
| `ControlMappingEngine` | `modules/lambda` | Triggered by EventBridge |

### DDIA connections
- **Ch. 1 (Reliability):** Automation prevents human error in infrastructure
- **Ch. 4 (Schema Evolution):** Adding GSIs = backward-compatible schema migration
- **Ch. 9 (Consistency):** Terraform state = leader-based source of truth
- **Ch. 11 (Idempotency):** `terraform apply` is idempotent, just like our pipeline functions

### DVA-C02 connections
- **CloudFormation:** Templates, !Ref, !GetAtt, Change Sets, StackSets, Drift Detection (Domain 4)
- **CodePipeline:** Source → Build → Deploy stages (Domain 4)
- **CodeBuild:** buildspec.yml, test + build (Domain 4)
- **CodeDeploy:** Lambda canary/linear deployment strategies (Domain 4)
- **CloudWatch:** Metrics, Alarms, Dashboards, X-Ray tracing (Domain 4)

**Next: Phase 7** — Demo environment, go-to-market, and running the full pipeline end-to-end.